# **HW2 – Entanglement, Bell States, and Query Algorithms**
_Time required: ~2–3 hours (for students with Qiskit basics)_

**What you’ll practice**
- Creating and verifying Bell states
- Measuring correlations and entanglement properties
- Implementing quantum teleportation and superdense coding
- Building oracles for query algorithms
- Deutsch-Jozsa and Bernstein-Vazirani algorithms
- Query complexity and speedups
- Connections to AI/DS applications

**What to turn in**
- This single notebook (`HW2_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later).
- Use the provided AerSimulator for reproducibility.
- If stuck, explain reasoning; partial credit for clear work.
- For circuits, use `qc.draw('mpl')`; for states, use `Statevector` or `DensityMatrix`.
- Pay attention to Qiskit’s little-endian for measurements.


In [ ]:
# --- Setup (run me first) ---
# Install Qiskit if needed (uncomment in Colab)
# !pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib

# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_histogram, plot_state_city
from qiskit.quantum_info import Statevector, DensityMatrix, Operator, concurrence
import numpy as np
import matplotlib.pyplot as plt

# Simulator backend
sim = AerSimulator()

# Function to get unitary matrix of a circuit
def unitary_of(circ):
    """Return the unitary matrix of the circuit."""
    return Operator(circ).data

# Function to run circuit and get counts
def get_counts(circ, shots=2000):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-8):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close:\n{A}\nvs\n{B}")


## Part A — Bell States & Entanglement (≈25 min)

**A1.** Build a circuit for |Φ+⟩ and draw it. Compute its statevector and verify.  
**A2.** Measure in computational basis (shots=2000), plot histogram, explain correlations.  
**A3.** Compute reduced density matrix for qubit 0; plot city diagram. Compute concurrence and confirm maximal entanglement.


In [ ]:
# A1. |Φ+⟩ circuit
qc_phi_plus = QuantumCircuit(2)
# YOUR CODE HERE: H on 0, CNOT 0 to 1
qc_phi_plus.draw('mpl')
plt.show()
sv_phi_plus = Statevector(qc_phi_plus)
print("Statevector: ", sv_phi_plus)
assert_close(sv_phi_plus.data, np.array([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)]))

# A2. Measure and plot
qc_phi_plus_meas = qc_phi_plus.copy()
qc_phi_plus_meas.measure_all()
counts_phi = None  # YOUR CODE HERE: use get_counts
plot_histogram(counts_phi)
plt.show()
# Written answer: Explain correlations (e.g., always same bits, 50/50 '00'/'11')

# A3. Reduced density and concurrence
rho_phi = DensityMatrix(sv_phi_plus)
rho_0 = rho_phi.partial_trace([1])  # Trace out qubit 1
plot_state_city(rho_0)
plt.show()
conc = concurrence(rho_phi)
print("Concurrence: ", conc)
assert_close(conc, 1.0)  # Maximal
# Written answer: Why is ρ_0 mixed? (Indicates entanglement)


## Part B — Quantum Teleportation (≈30 min)

**B1.** Implement a teleportation circuit: Prepare |ψ⟩ on Alice's qubit 0 (e.g., RY(π/3)), create Bell pair on 1 and 2, Bell measurement on 0-1, conditional X/Z on Bob's 2 based on classical bits.  
**B2.** Evolve statevector (no measurement) and verify Bob's qubit matches original |ψ⟩.  
**B3.** Short answer: Why does this not violate no-cloning? How does entanglement enable it?


In [ ]:
# B1. Teleportation circuit (3 qubits: 0=psi, 1=Alice half, 2=Bob)
qc_tele = QuantumCircuit(3, 2)
# YOUR CODE HERE: Prepare |ψ> on 0 (e.g., ry(np.pi/3, 0))
# Create Bell on 1-2: h(1), cx(1,2)
# Bell measurement: cx(0,1), h(0), measure 0 and 1 to cbits 0 and 1
# Bob corrections: x(2) if c1==1, z(2) if c0==1 (use x_if, z_if)
qc_tele.draw('mpl')
plt.show()

# B2. Evolve statevector (simulate without actual measurement)
sv_tele = Statevector(qc_tele)  # But remove measures for statevector
# To simulate full: use conditional gates properly
# For verification, compute final state on qubit 2
rho_bob = sv_tele.partial_trace([0,1])
print("Bob's reduced density: ", rho_bob)
# Written answer: Confirm it matches original |ψ><ψ|

# B3. Written answer: (No-cloning: original destroyed; entanglement provides correlation channel)


## Part C — Superdense Coding (≈25 min)

**C1.** Build superdense coding circuit for message '10': Create Bell on 0-1, Alice applies Z on 0 (for '10'), sends to Bob; Bob does CNOT 0 to 1, H on 0, measure both.  
**C2.** Run (shots=1) and confirm measurement yields '10'.  
**C3.** Short answer: How does this send 2 bits with 1 qubit? Role of entanglement?


In [ ]:
# C1. Superdense circuit for '10'
qc_sd = QuantumCircuit(2, 2)
# YOUR CODE HERE: Bell on 0-1 (h(0), cx(0,1))
# Alice: for '10' apply z(0)
# Bob: cx(0,1), h(0), measure_all
qc_sd.draw('mpl')
plt.show()

# C2. Run and check
counts_sd = get_counts(qc_sd, shots=1)
print("Measurement: ", counts_sd)
# Assert it's '10' (but in Qiskit bitstring, little-endian: check '01' for '10'? Wait, adjust for ordering)

# C3. Written answer: (Encodes in Paulis; entanglement allows Bob to decode 2 bits from 1 qubit via shared pair)


## Part D — Deutsch-Jozsa Algorithm (≈30 min)

**D1.** Implement a balanced oracle for n=3 (e.g., f(x)=x0 XOR x1 XOR x2). Build DJ circuit.  
**D2.** Run (shots=1) and check measurement ≠ '000' (balanced).  
**D3.** Implement constant oracle, run, confirm '000'. Short answer: Why 1 query suffices quantum vs classical exponential.


In [ ]:
# D1. Balanced oracle and DJ circuit (n=3)
n = 3
qc_oracle_bal = QuantumCircuit(n+1)
# YOUR CODE HERE: e.g., cx(0,n), cx(1,n), cx(2,n) for XOR
qc_dj_bal = QuantumCircuit(n+1, n)
# Add H to all, x(n) then H(n) for |->, oracle, H to first n, measure first n
qc_dj_bal = qc_dj_bal.compose(qc_oracle_bal, range(n+1))
qc_dj_bal.draw('mpl')
plt.show()

# D2. Run balanced
counts_dj_bal = get_counts(qc_dj_bal, shots=1)
print("Balanced result: ", counts_dj_bal)
# Assert not '000'

# D3. Constant oracle
qc_oracle_const = QuantumCircuit(n+1)  # Do nothing for constant 0
qc_dj_const = QuantumCircuit(n+1, n)
# Similar composition
counts_dj_const = get_counts(qc_dj_const, shots=1)
print("Constant result: ", counts_dj_const)
# Written answer: (Superposition + interference extracts global property in 1 query)


## Part E — Bernstein-Vazirani Algorithm (≈25 min)

**E1.** Implement oracle for hidden s='101' (n=3): f(x)=s·x mod 2. Build BV circuit.  
**E2.** Run (shots=1) and confirm measurement '101'.  
**E3.** Short answer: Speedup over classical? Potential AI application (e.g., hidden linear pattern).


In [ ]:
# E1. BV oracle for s='101'
qc_oracle_bv = QuantumCircuit(n+1)
# YOUR CODE HERE: cx(i,n) for each i where s_i=1
qc_bv = QuantumCircuit(n+1, n)
# Similar to DJ: H all, |-> on anc, oracle, H first n, measure first n
qc_bv = qc_bv.compose(qc_oracle_bv, range(n+1))
qc_bv.draw('mpl')
plt.show()

# E2. Run
counts_bv = get_counts(qc_bv, shots=1)
print("BV result: ", counts_bv)
# Assert '101' (or reverse if endian)

# E3. Written answer: (1 vs n queries; e.g., learn linear features in data)
